# California Housing Explainability Project

In this notebook, I analyze the California Housing dataset using a Gradient Boosting Regressor.
The goal is to build a predictive model for median house value and then use explainability techniques
to understand how the model uses features.

Techniques used:
- **Partial Dependence Plots (PDP)**  
- **Individual Conditional Expectation (ICE)**  
- **Accumulated Local Effects (ALE)**  

Along the way, I also perform exploratory data analysis (EDA) to check feature correlations,
since correlations can strongly affect interpretation of PDP vs ALE.

In [ ]:
# =======================================
# Part 1: Imports
# =======================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.inspection import PartialDependenceDisplay

np.random.seed(42)

In [ ]:
# =======================================
# Part 2: Load Data
# =======================================
housing = fetch_california_housing(as_frame=True)
df = housing.frame.copy()
display(df.head())
display(df.describe())

# Target and features
X = df.drop('MedHouseVal', axis=1)
y = df['MedHouseVal']

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
X_train.shape, X_test.shape

## Exploratory Data Analysis (EDA)

The correlation heatmap helps identify relationships among predictors and between predictors and the target (for a quick check, we append the target to the matrix temporarily).

**What to look for:**
- Strong positive correlation between `MedInc` and `MedHouseVal`.
- Geographic variables (`Latitude`, `Longitude`) often show structure with the target.
- Some predictors are correlated with each other (e.g., occupancy/rooms), which matters for PDP interpretation.

In [ ]:
# =======================================
# Part 3: Correlation Heatmap (Matplotlib only)
# =======================================
# Build correlation including target for visualization
corr_df = df.copy()
corr = corr_df.corr(numeric_only=True)

fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(corr.values, interpolation='nearest')
ax.set_xticks(range(len(corr.columns)))
ax.set_yticks(range(len(corr.index)))
ax.set_xticklabels(corr.columns, rotation=45, ha='right')
ax.set_yticklabels(corr.index)
ax.set_title('Correlation Heatmap')
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()

# Histograms for a few features
for col in ['MedInc', 'HouseAge', 'AveRooms', 'AveOccup']:
    plt.figure()
    plt.hist(df[col], bins=40)
    plt.title(f'Distribution of {col}')
    plt.xlabel(col)
    plt.ylabel('Count')
    plt.show()

## Model Training Results

I train a **Gradient Boosting Regressor** and evaluate it with R² and MSE.

In [ ]:
# =======================================
# Part 4: Train Gradient Boosting Regressor
# =======================================
gbr = GradientBoostingRegressor(random_state=42)
gbr.fit(X_train, y_train)

y_pred = gbr.predict(X_test)
r2 = r2_score(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
print(f'R2 Score: {r2:.3f}')
print(f'MSE: {mse:.3f}')

## Partial Dependence Plots (PDP)

PDPs show **average** model response when a feature is varied over its range.
They are global and can be biased when features are correlated.

In [ ]:
# =======================================
# Part 5: PDP and ICE (Matplotlib via sklearn)
# =======================================
features_to_plot = ['MedInc', 'AveRooms', 'HouseAge']
fig, ax = plt.subplots(figsize=(10, 5))
PartialDependenceDisplay.from_estimator(
    gbr, X_train, features_to_plot, kind='average', ax=ax
)
plt.suptitle('Partial Dependence Plots (PDP)')
plt.tight_layout()
plt.show()

# ICE for MedInc
fig, ax = plt.subplots(figsize=(10, 5))
PartialDependenceDisplay.from_estimator(
    gbr, X_train, ['MedInc'], kind='individual', subsample=80, random_state=42, ax=ax
)
plt.suptitle('ICE Plot for MedInc')
plt.tight_layout()
plt.show()

## Accumulated Local Effects (ALE)

ALE adjusts for **feature correlations** by computing local effects within bins of the feature and accumulating them.
Below is a lightweight 1D ALE implementation (Apley & Zhu, 2019) so the notebook has **no external dependencies**.

In [ ]:
# =======================================
# Part 6: Simple 1D ALE implementation
# =======================================
def ale_1d(model, X: pd.DataFrame, feature: str, bins=20, random_state=42):
    """Compute 1D ALE for a single feature.
    Returns bin centers and ALE values centered to have mean zero.
    """
    rng = np.random.RandomState(random_state)
    x = X[feature].values
    # quantile-based bin edges
    edges = np.quantile(x, np.linspace(0, 1, bins + 1))
    # ensure unique edges
    edges = np.unique(edges)
    bins_eff = len(edges) - 1
    effects = np.zeros(bins_eff)
    counts = np.zeros(bins_eff)

    for i in range(bins_eff):
        lo, hi = edges[i], edges[i+1]
        mask = (x >= lo) & (x < hi) if i < bins_eff - 1 else (x >= lo) & (x <= hi)
        idx = np.where(mask)[0]
        if idx.size == 0:
            continue
        X_lo = X.iloc[idx].copy()
        X_hi = X.iloc[idx].copy()
        X_lo[feature] = lo
        X_hi[feature] = hi
        pred_lo = model.predict(X_lo)
        pred_hi = model.predict(X_hi)
        effects[i] = np.mean(pred_hi - pred_lo)
        counts[i] = idx.size

    # Accumulate local effects
    ale_vals = np.cumsum(effects)
    # Centering to zero mean
    # Weighted midpoints for step function centering
    centers = (edges[:-1] + edges[1:]) / 2.0
    # compute step function to sample-average and subtract
    # Expand step function to observations
    # Map each x to its bin index
    bin_idx = np.minimum(np.searchsorted(edges, x, side='right') - 1, bins_eff - 1)
    bin_idx = np.maximum(bin_idx, 0)
    centered = ale_vals[bin_idx]
    mean_center = centered.mean()
    ale_vals = ale_vals - mean_center
    return centers, ale_vals

def plot_ale_1d(model, X, feature, bins=20):
    centers, ale_vals = ale_1d(model, X, feature, bins=bins)
    plt.figure(figsize=(8,4))
    plt.plot(centers, ale_vals)
    plt.xlabel(feature)
    plt.ylabel('ALE')
    plt.title(f'ALE for {feature}')
    plt.tight_layout()
    plt.show()

for feat in ['MedInc', 'AveRooms', 'HouseAge']:
    plot_ale_1d(gbr, X_train, feat, bins=20)

## Discussion: PDP vs. ICE vs. ALE (Draft Notes)

- **PDP** (global average) shows clear positive impact of `MedInc` on predicted house value; effects for `AveRooms` and `HouseAge` are milder.
- **ICE** reveals heterogeneity for `MedInc`—some households gain value faster than others as income rises.
- **ALE** curves are slightly less extreme in high-income ranges, suggesting PDP may overstate effects there due to correlation with geography.

These methods are **complementary**: PDP gives an overall trend, ICE shows individual variation, and ALE accounts for correlated features.

# Conclusion

- **EDA** showed strong correlation between income and house value, plus moderate correlations among other predictors.
- **PDP** highlighted the main global trends but can be biased by correlation.
- **ICE** exposed individual-level variation around the average trends.
- **ALE** corrected for correlation to provide a more realistic local effect estimate.

Together, these perspectives give a well-rounded view of the model's behavior.